## Constants
- land-sea mask
- orography
## Surface Variabels
- two meter temperature
- total column water vapor

In [55]:
import netCDF4 as nc
import healpy as hp
import numpy as np
from scipy.interpolate import griddata

In [56]:
dataset = nc.Dataset('/Users/annelouisedeboer/Desktop/Thesis_prep/dlwp-hpx/src/dlwp-hpx/data/era5_z500.nc', 'r')

In [57]:
dataset

<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    CDI: Climate Data Interface version 2.1.0 (https://mpimet.mpg.de/cdi)
    Conventions: CF-1.6
    history: Tue Nov 22 07:21:36 2022: cdo -b F32 mergetime 1979-2021_era5_1deg_3h_geopotential_500_p0.nc 1979-2021_era5_1deg_3h_geopotential_500_p1.nc 1979-2021_era5_1deg_3h_geopotential_500.nc
2022-10-27 23:47:30 GMT by grib_to_netcdf-2.25.1: /opt/ecmwf/mars-client/bin/grib_to_netcdf.bin -S param -o /cache/data9/adaptor.mars.internal-1666913232.3154385-28517-1-07242098-fb0a-498f-9e07-2bc046b9f5fe.nc /cache/tmp/07242098-fb0a-498f-9e07-2bc046b9f5fe-adaptor.mars.internal-1666910096.397881-28517-1-tmp.grib
    CDO: Climate Data Operators version 2.1.0 (https://mpimet.mpg.de/cdo)
    dimensions(sizes): time(16), longitude(360), latitude(181), level(1)
    variables(dimensions): int32 time(time), float32 longitude(longitude), float32 latitude(latitude), float32 level(level), float32 z(time, level, latitude, longitude)

In [58]:
dataset.variables['time']

<class 'netCDF4.Variable'>
int32 time(time)
    standard_name: time
    long_name: time
    axis: T
    units: hours since 1900-01-01
    calendar: gregorian
unlimited dimensions: time
current shape = (16,)
filling on, default _FillValue of -2147483647 used

In [62]:
import numpy as np
import pandas as pd
import xarray as xr
# Example masked array (replace with your actual data)
masked_array = np.ma.masked_array(data=[1034376, 1034379, 1034382, 1034385, 
                                         1034388, 1034391, 1034394, 1034397,
                                         1034400, 1034403, 1034406, 1034409,
                                         1034412, 1034415, 1034418, 1034421],
                                   mask=False)

# Convert hours since 1900-01-01 to datetime
reference_date = pd.Timestamp('1900-01-01')
time_dates = reference_date + pd.to_timedelta(masked_array, unit='h')

# Print converted dates for verification
print("Converted Time Dates:")
print(time_dates)

Converted Time Dates:
DatetimeIndex(['2018-01-01 00:00:00', '2018-01-01 03:00:00',
               '2018-01-01 06:00:00', '2018-01-01 09:00:00',
               '2018-01-01 12:00:00', '2018-01-01 15:00:00',
               '2018-01-01 18:00:00', '2018-01-01 21:00:00',
               '2018-01-02 00:00:00', '2018-01-02 03:00:00',
               '2018-01-02 06:00:00', '2018-01-02 09:00:00',
               '2018-01-02 12:00:00', '2018-01-02 15:00:00',
               '2018-01-02 18:00:00', '2018-01-02 21:00:00'],
              dtype='datetime64[ns]', freq=None)


In [59]:
lats = dataset.variables['latitude'][:]
lons = dataset.variables['longitude'][:]
levels = dataset.variables['level'][:]
times = dataset.variables['time'][:]
z = dataset.variables['z'][:]

In [60]:
times

masked_array(data=[1034376, 1034379, 1034382, 1034385, 1034388, 1034391,
                   1034394, 1034397, 1034400, 1034403, 1034406, 1034409,
                   1034412, 1034415, 1034418, 1034421],
             mask=False,
       fill_value=np.int64(999999),
            dtype=int32)

In [ ]:
time_index = 10
level_index = 0
data = z[time_index, level_index, :, :]

In [ ]:
nside = 128  # Choose an appropriate HEALPix resolution
npix = hp.nside2npix(nside)

# Create arrays of HEALPix pixel coordinates
theta, phi = hp.pix2ang(nside, np.arange(npix))

# Convert to lat/lon in degrees
hp_lats = 90 - np.degrees(theta)
hp_lons = np.degrees(phi)

# Create a 2D grid of the original lat/lon coordinates
lons_grid, lats_grid = np.meshgrid(lons, lats)

# Interpolate the original data onto the HEALPix grid
healpix_map = griddata((lons_grid.flatten(), lats_grid.flatten()), data.flatten(), (hp_lons, hp_lats), method='linear')

In [ ]:
print(len(data.flatten()))
print(len(lons_grid.flatten()))

In [ ]:
hp.cartview(healpix_map, title="ERA5 Geopotential Data", unit="m**2 s**-2")

In [ ]:
hp.mollview(healpix_map, title="ERA5 Geopotential Data", unit="m**2 s**-2")
hp.graticule()


In [ ]:
#import healpix format data: /Users/annelouisedeboer/Desktop/Thesis_prep/dlwp-hpx/src/dlwp-hpx/era5_1deg_3h_HPX32_1979-2018_z500.nc

In [ ]:
# Load the NetCDF file
file_path = '/Users/annelouisedeboer/Desktop/Thesis_prep/dlwp-hpx/src/dlwp-hpx/era5_1deg_3h_HPX32_1979-2018_z500.nc'
 # Load .nc file in HEALPix format to get nside information and to initialize the remapper module


latitudes=181,
longitudes=360,
nside=32
fc_ds_hpx = xr.open_dataset(file_path)
fc_ds_hpx # face: 12 height: 32 width: 32 sample: 16 level: 1

In [72]:
# Set environment variables
!export WORLD_RANK=0
!export WORLD_SIZE=0
!export HDF5_USE_FILE_LOCKING=True
!export CUDA_VISIBLE_DEVICES=""  # Set this according to your GPU configuration
!export HYDRA_FULL_ERROR=1  

# Run the Python script with arguments
!python -u /Users/annelouisedeboer/Desktop/Thesis_prep/dlwp-hpx/src/dlwp-hpx/scripts/train.py num_workers=8 port=29450 learning_rate=2e-4 batch_size=16 experiment_name=hpx34_unet_ANNE model=hpx_rec_unet model/modules/blocks@model.encoder.conv_block=conv_next_block model/modules/blocks@model.decoder.conv_block=conv_next_block model.encoder.n_channels='[136,68,34]' model.decoder.n_channels='[34,68,136]' trainer.max_epochs=300 data=era5_hpx32_1var_3h_24h data.prefix=era5_1deg_3h_HPX32_1979-2018_ data.prebuilt_dataset=False data.module.drop_last=True trainer/lr_scheduler=cosine trainer/optimizer=adam

[2024-10-26 20:49:28,954][HYDRA] Hydra 1.3.2
[2024-10-26 20:49:28,954][HYDRA] ===========
[2024-10-26 20:49:28,954][HYDRA] Installed Hydra Plugins
[2024-10-26 20:49:28,954][HYDRA] ***********************
[2024-10-26 20:49:28,954][HYDRA] 	ConfigSource:
[2024-10-26 20:49:28,954][HYDRA] 	-------------
[2024-10-26 20:49:28,955][HYDRA] 		FileConfigSource
[2024-10-26 20:49:28,955][HYDRA] 		ImportlibResourcesConfigSource
[2024-10-26 20:49:28,955][HYDRA] 		StructuredConfigSource
[2024-10-26 20:49:28,955][HYDRA] 	CompletionPlugin:
[2024-10-26 20:49:28,955][HYDRA] 	-----------------
[2024-10-26 20:49:28,955][HYDRA] 		BashCompletion
[2024-10-26 20:49:28,955][HYDRA] 		FishCompletion
[2024-10-26 20:49:28,955][HYDRA] 		ZshCompletion
[2024-10-26 20:49:28,955][HYDRA] 	Launcher:
[2024-10-26 20:49:28,955][HYDRA] 	---------
[2024-10-26 20:49:28,955][HYDRA] 		BasicLauncher
[2024-10-26 20:49:28,955][HYDRA] 	Sweeper:
[2024-10-26 20:49:28,955][HYDRA] 	--------
[2024-10-26 20:49:28,955][HYDRA] 		BasicSweeper


In [10]:
import xarray as xr

data = xr.open_dataset('src/dlwp-hpx/data/era5_1deg_3h_HPX32_1979-2018_z500.nc', autoclose=True)

In [39]:
import xarray as xr
data = xr.open_dataset('/Users/annelouisedeboer/Desktop/Thesis_prep/dlwp-hpx/merged_dataset.nc', autoclose=True)
#print(data)
#print(data[['z']].dims)  # Check the existing dimensions
data[['z']] #.to_array('channel_in', name='inputs').transpose( 'time', 'channel_in', 'face', 'height', 'width', 'level')


<xarray.Dataset> Size: 984kB
Dimensions:  (time: 16, level: 1, face: 12, height: 32, width: 32)
Coordinates:
    lat      (face, height, width) float64 98kB ...
    lon      (face, height, width) float64 98kB ...
  * time     (time) datetime64[ns] 128B 2018-01-01 ... 2018-01-02T21:00:00
  * level    (level) float32 4B 500.0
  * face     (face) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
  * height   (height) int64 256B 0 1 2 3 4 5 6 7 8 ... 24 25 26 27 28 29 30 31
  * width    (width) int64 256B 0 1 2 3 4 5 6 7 8 ... 23 24 25 26 27 28 29 30 31
Data variables:
    z        (time, level, face, height, width) float32 786kB ...
Attributes:
    CDI:          Climate Data Interface version 2.1.0 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Tue Nov 22 07:21:36 2022: cdo -b F32 mergetime 1979-2021_er...
    CDO:          Climate Data Operators version 2.1.0 (https://mpimet.mpg.de...

In [11]:
data[list('z500')]

KeyError: '5'